<a id="s9"></a>
---
## Section 9 — Capstone: Library Management System

This capstone brings together everything from the unit:
- Creating multiple related tables
- INSERT, SELECT, UPDATE, DELETE
- Transactions (borrow / return operations)
- Error handling

### Database Design

```
┌─────────────────────┐    ┌─────────────────────┐
│       books         │    │      members        │
├─────────────────────┤    ├─────────────────────┤
│ book_id  (PK)       │    │ member_id (PK)      │
│ title    TEXT       │    │ name      TEXT      │
│ author   TEXT       │    │ email     UNIQUE     │
│ copies   INTEGER    │    │ joined    TEXT      │
│ available INTEGER   │    └──────────┬──────────┘
└──────────┬──────────┘               │
           └─────────────┬────────────┘
                         ▼
           ┌─────────────────────┐
           │        loans        │
           ├─────────────────────┤
           │ loan_id   (PK)      │
           │ book_id   → books   │
           │ member_id → members │
           │ loan_date TEXT      │
           │ return_date TEXT    │
           └─────────────────────┘
```

The `loans` table links books to members. No book title or member name is repeated — foreign keys do the linking. This avoids data duplication (a principle called **normalisation**).

In [ ]:
# ── Capstone — Database setup ────────────────────────────────────────────

import sqlite3

def setup_library():
    """Create and return an in-memory library database."""
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS books (
            book_id   INTEGER PRIMARY KEY AUTOINCREMENT,
            title     TEXT    NOT NULL,
            author    TEXT    NOT NULL,
            copies    INTEGER NOT NULL DEFAULT 1,
            available INTEGER NOT NULL DEFAULT 1,
            CHECK (available >= 0)
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS members (
            member_id INTEGER PRIMARY KEY AUTOINCREMENT,
            name      TEXT    NOT NULL,
            email     TEXT    UNIQUE NOT NULL,
            joined    TEXT    NOT NULL
        )
    """)

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS loans (
            loan_id     INTEGER PRIMARY KEY AUTOINCREMENT,
            book_id     INTEGER NOT NULL,
            member_id   INTEGER NOT NULL,
            loan_date   TEXT    NOT NULL,
            return_date TEXT
        )
    """)

    conn.commit()
    return conn

LIBRARY_DB = setup_library()
print('Library database ready.')

In [ ]:
# ── Capstone — Seed data ─────────────────────────────────────────────────

def seed_library(conn):
    cursor = conn.cursor()

    cursor.executemany(
        'INSERT INTO books (title, author, copies, available) VALUES (?,?,?,?)',
        [
            ('Python Crash Course',         'Eric Matthes',    3, 3),
            ('Automate the Boring Stuff',   'Al Sweigart',     2, 2),
            ('Clean Code',                 'Robert Martin',   1, 1),
            ('The Pragmatic Programmer',   'Hunt & Thomas',   2, 2),
        ]
    )

    cursor.executemany(
        'INSERT INTO members (name, email, joined) VALUES (?,?,?)',
        [
            ('Alice Smith', 'alice@library.com', '2024-09-01'),
            ('Bob Jones',   'bob@library.com',   '2024-09-05'),
            ('Carol Brown', 'carol@library.com', '2024-09-10'),
        ]
    )
    conn.commit()
    print('Seed data inserted.')

seed_library(LIBRARY_DB)

# Show catalogue
cursor = LIBRARY_DB.cursor()
cursor.execute('SELECT book_id, title, available, copies FROM books')
print('\nBook catalogue:')
for row in cursor.fetchall():
    print(f'  [{row[0]}] {row[1]:40s}  available: {row[2]}/{row[3]}')

In [ ]:
# ── Capstone — borrow_book() with transaction ─────────────────────────────

def borrow_book(conn, member_id, book_id, loan_date='2024-10-15'):
    """Borrow a book: update availability + create loan record (transaction)."""
    cursor = conn.cursor()
    try:
        # Check availability
        cursor.execute(
            'SELECT title, available FROM books WHERE book_id = ?',
            (book_id,)
        )
        row = cursor.fetchone()
        if not row:
            print(f'  Book {book_id} not found.')
            return False
        title, available = row
        if available == 0:
            print(f'  "{title}" is not available.')
            return False

        # Two operations inside one transaction
        cursor.execute(
            'UPDATE books SET available = available - 1 WHERE book_id = ?',
            (book_id,)
        )
        cursor.execute(
            'INSERT INTO loans (book_id, member_id, loan_date) VALUES (?, ?, ?)',
            (book_id, member_id, loan_date)
        )
        conn.commit()    # save both operations together
        print(f'  Borrowed: "{title}"  (loan id: {cursor.lastrowid})')
        return True

    except sqlite3.Error as e:
        conn.rollback()  # undo both operations on any error
        print(f'  Error: {e}')
        return False

print('Borrowing books:')
borrow_book(LIBRARY_DB, member_id=1, book_id=1)  # Alice borrows Python Crash Course
borrow_book(LIBRARY_DB, member_id=2, book_id=1)  # Bob borrows same book
borrow_book(LIBRARY_DB, member_id=3, book_id=3)  # Carol borrows Clean Code
borrow_book(LIBRARY_DB, member_id=1, book_id=3)  # Alice tries — already out!

# Show updated availability
cursor = LIBRARY_DB.cursor()
cursor.execute('SELECT title, available, copies FROM books')
print('\nUpdated catalogue:')
for title, avail, copies in cursor.fetchall():
    print(f'  {title:40s}  {avail}/{copies}')

In [ ]:
# ── Capstone — return_book() ─────────────────────────────────────────────

def return_book(conn, loan_id, return_date='2024-10-22'):
    """Return a borrowed book: update availability + set return date."""
    cursor = conn.cursor()
    try:
        # Find the loan
        cursor.execute(
            'SELECT book_id, return_date FROM loans WHERE loan_id = ?',
            (loan_id,)
        )
        row = cursor.fetchone()
        if not row:
            print(f'  Loan {loan_id} not found.')
            return False
        book_id, existing_return = row
        if existing_return:
            print(f'  Loan {loan_id} already returned on {existing_return}.')
            return False

        # Two operations inside one transaction
        cursor.execute(
            'UPDATE loans SET return_date = ? WHERE loan_id = ?',
            (return_date, loan_id)
        )
        cursor.execute(
            'UPDATE books SET available = available + 1 WHERE book_id = ?',
            (book_id,)
        )
        conn.commit()
        print(f'  Loan {loan_id} returned on {return_date}.')
        return True

    except sqlite3.Error as e:
        conn.rollback()
        print(f'  Error: {e}')
        return False

print('Returning books:')
return_book(LIBRARY_DB, loan_id=1)  # Alice returns Python Crash Course
return_book(LIBRARY_DB, loan_id=1)  # Try to return again — already returned

cursor = LIBRARY_DB.cursor()
cursor.execute('SELECT loan_id, book_id, member_id, loan_date, return_date FROM loans')
print('\nLoan records:')
for row in cursor.fetchall():
    print(f'  {row}')

In [ ]:
# ── Capstone — search and report ─────────────────────────────────────────

def search_books(conn, keyword):
    """Search books by title or author keyword."""
    cursor = conn.cursor()
    cursor.execute(
        'SELECT book_id, title, author, available FROM books'
        ' WHERE title LIKE ? OR author LIKE ?',
        (f'%{keyword}%', f'%{keyword}%')
    )
    results = cursor.fetchall()
    if not results:
        print(f'  No books matching "{keyword}".')
    for book_id, title, author, avail in results:
        status = 'available' if avail > 0 else 'all out'
        print(f'  [{book_id}] {title} by {author}  — {status}')

def active_loans_report(conn):
    """Show all currently borrowed books."""
    cursor = conn.cursor()
    cursor.execute("""
        SELECT m.name, b.title, l.loan_date
        FROM loans l
        JOIN books   b ON l.book_id   = b.book_id
        JOIN members m ON l.member_id = m.member_id
        WHERE l.return_date IS NULL
        ORDER BY l.loan_date
    """)
    rows = cursor.fetchall()
    if not rows:
        print('  No active loans.')
    for member, title, date in rows:
        print(f'  {member:15s} — "{title}"  (since {date})')

print('Search results for "Python":')
search_books(LIBRARY_DB, 'Python')

print('\nSearch results for "Matthes":')
search_books(LIBRARY_DB, 'Matthes')

print('\nActive loans:')
active_loans_report(LIBRARY_DB)

LIBRARY_DB.close()
print('\nLibrary database closed.')

<a id="practice"></a>
---
## Practice Tasks

Complete each task in the code cell below it.  
Use `:memory:` databases so your work doesn't leave files on disk.

### Practice Task 1 — Product Catalogue

Create a database for a small shop:

1. Create a `products` table with columns: `product_id` (PK, AUTOINCREMENT), `name` (TEXT NOT NULL), `price` (REAL NOT NULL), `stock` (INTEGER DEFAULT 0), `CHECK (price > 0)`, `CHECK (stock >= 0)`
2. Insert at least 5 products using `executemany()`
3. Display all products ordered by price ascending
4. Print the average price, most expensive, and cheapest product
5. Update the stock of one product
6. Delete a product whose stock is 0

In [ ]:
# Practice Task 1 — Product Catalogue
# Write your solution here.

import sqlite3

# Your code...


### Practice Task 2 — High-Score Table

Build a game leaderboard:

1. Create a `scores` table with: `score_id` (PK), `player_name` (TEXT NOT NULL), `score` (INTEGER NOT NULL), `level` (INTEGER), `date_played` (TEXT)
2. Insert at least 8 scores for different players (some players can have multiple scores)
3. Display the **top 5** scores in descending order
4. For each player, show their **best** score using `GROUP BY` and `MAX()`
5. Show only players whose best score is above 5000

In [ ]:
# Practice Task 2 — High-Score Table
# Write your solution here.

import sqlite3

# Hint: GROUP BY groups rows with the same value
# SELECT player_name, MAX(score) FROM scores GROUP BY player_name

# Your code...


### Practice Task 3 — Student Grade Tracker

Build a grade tracking system:

1. Create two tables: `students` (id, name, year) and `grades` (grade_id, student_id, subject, mark, semester)
2. Insert 3 students and at least 12 grade records
3. For each student, calculate and display their average mark across all subjects
4. Find the student with the highest average
5. Update all marks in a specific subject by adding 5 (bonus marks)
6. Delete all grades with a mark below 30 (fail threshold)
7. Wrap steps 5 and 6 in a single transaction

In [ ]:
# Practice Task 3 — Student Grade Tracker
# Write your solution here.

import sqlite3

# Your code...


<a id="debug"></a>
---
## Debugging Exercises

Each cell below contains **intentional bugs**. Find and fix each bug, then run the cell to confirm it works.

> **Tip:** Read the error message carefully — it usually tells you exactly what went wrong.

### Debug Exercise 1 — SQL Injection Vulnerability

The code below has a critical security bug. Find and fix it.

In [ ]:
# Debug Exercise 1
# FIND AND FIX the SQL injection vulnerability in this code.

import sqlite3

def find_student(name):
    with sqlite3.connect(':memory:') as conn:
        cursor = conn.cursor()
        cursor.execute(
            'CREATE TABLE students (id INTEGER, name TEXT, gpa REAL)'
        )
        cursor.execute(
            "INSERT INTO students VALUES (1, 'Alice', 3.8)"
        )
        conn.commit()

        # ❌ BUG: This is vulnerable to SQL injection!
        query = "SELECT * FROM students WHERE name = '" + name + "'"
        cursor.execute(query)
        return cursor.fetchall()

# Test your fix:
print(find_student('Alice'))
# Expected: [(1, 'Alice', 3.8)]


### Debug Exercise 2 — Missing Trailing Comma

This code raises a `TypeError`. Find and fix the bug.

In [ ]:
# Debug Exercise 2
# FIND AND FIX the error causing the TypeError.

import sqlite3

with sqlite3.connect(':memory:') as conn:
    cursor = conn.cursor()
    cursor.execute(
        'CREATE TABLE items (id INTEGER, name TEXT)'
    )
    cursor.execute(
        'INSERT INTO items VALUES (1, ?)',
        ('Laptop')    # ❌ BUG IS HERE
    )
    conn.commit()
    cursor.execute('SELECT * FROM items')
    print(cursor.fetchall())
    # Expected: [(1, 'Laptop')]


### Debug Exercise 3 — Missing WHERE Clause

This UPDATE statement has a dangerous mistake. Fix it so only Bob's GPA changes.

In [ ]:
# Debug Exercise 3
# FIND AND FIX the dangerous mistake in this UPDATE statement.

import sqlite3

conn = make_students_db()   # reuse our helper from Section 5
cursor = conn.cursor()

print('Before update:')
cursor.execute('SELECT name, gpa FROM students')
print(cursor.fetchall())

# ❌ BUG: This changes EVERY student's GPA!
cursor.execute('UPDATE students SET gpa = 1.0')
conn.commit()

print('After update:')
cursor.execute('SELECT name, gpa FROM students')
print(cursor.fetchall())
# Expected: only Bob's GPA should be 1.0

conn.close()


<a id="challenge"></a>
---
## Mini Challenges

These challenges require combining multiple skills from the unit.

### Mini Challenge 1 — Shopping Cart System

Build a working shopping cart:

**Tables:**
- `products` (product_id, name, price, stock)
- `cart` (cart_id, product_id, quantity, added_at)

**Functions to implement:**
1. `add_to_cart(conn, product_id, quantity)` — adds item,    rejects if stock < quantity, updates stock
2. `remove_from_cart(conn, cart_id)` — removes item, restores stock
3. `show_cart(conn)` — displays cart with name, quantity, and line total
4. `checkout(conn)` — prints total, clears cart

Wrap `add_to_cart` operations in a **transaction** — both the cart insert AND the stock update must succeed together.

**Test your system with at least 3 products and multiple add/remove operations.**

In [ ]:
# Mini Challenge 1 — Shopping Cart System
# Implement the four functions described above.

import sqlite3

# Your code...


### Mini Challenge 2 — Contact Book

Build a command-line contact book application:

**Table:** `contacts` (contact_id, first_name, last_name, phone, email UNIQUE, group_name)

**Functions to implement:**
1. `add_contact(conn, first, last, phone, email, group)` — with duplicate email handling
2. `search_contact(conn, query)` — search by any part of name, phone, or email
3. `update_phone(conn, email, new_phone)` — update phone by email
4. `delete_contact(conn, email)` — delete by email
5. `list_group(conn, group_name)` — list all contacts in a group
6. `contact_stats(conn)` — total contacts, contacts per group

**Extra:** Handle `IntegrityError` gracefully in `add_contact`.

In [ ]:
# Mini Challenge 2 — Contact Book
# Implement the six functions described above.

import sqlite3

# Your code...
